# Training Script for a variety of different Time Series Models

## Settings and Imports

In [ ]:
from utils.get_features import get_matched_weather_load_data
from datetime import date
import pandas as pd
from dotenv import load_dotenv
import os

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

import joblib

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

FEATURES = ['Temp', 'Min Temp', 'Max Temp', 'Wind Speed',
       'Sunshine Duration', 'Cloud Cover', 'is_weekend', 'is_holiday',
       'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'lag_1',
       'lag_7', 'lag_14', 'rolling_mean_7']

TARGET = "load_mw"

SCALE_FEATURES = ['Temp', 'Min Temp', 'Max Temp', 'Wind Speed',
       'Sunshine Duration', 'Cloud Cover', 'lag_1',
       'lag_7', 'lag_14', 'rolling_mean_7']

## Helpers

In [ ]:
def evaluate_and_plot_model(model, X_test_scaled, y_test, test_df):
    y_pred_test = model.predict(X_test_scaled)

    plot_df = pd.DataFrame({
        "actual": y_test.values,
        "predicted": y_pred_test
    }, index=test_df.loc[y_test.index, "time"] if "time" in test_df.columns else y_test.index)
    plot_df = plot_df.sort_index()

    print(f"Test MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
    print(f"Test RMSE: {mean_squared_error(y_test, y_pred_test):.2f}")

    plt.figure(figsize=(14, 5))
    plt.plot(plot_df.index, plot_df["actual"], label="Actual", linewidth=1.8)
    plt.plot(plot_df.index, plot_df["predicted"], label="Predicted", linewidth=1.5, alpha=0.85)
    plt.title("Test Set: Predicted vs Actual Load")
    plt.xlabel("Date")
    plt.ylabel("Load (MW)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # plt.figure(figsize=(6, 6))
    # plt.scatter(plot_df["actual"], plot_df["predicted"], alpha=0.35)
    # axis_min = min(plot_df["actual"].min(), plot_df["predicted"].min())
    # axis_max = max(plot_df["actual"].max(), plot_df["predicted"].max())
    # plt.plot([axis_min, axis_max], [axis_min, axis_max], "r--", linewidth=1.2)
    # plt.title("Predicted vs Actual Scatter (Test)")
    # plt.xlabel("Actual Load (MW)")
    # plt.ylabel("Predicted Load (MW)")
    # plt.grid(alpha=0.3)
    # plt.tight_layout()
    # plt.show()

## Dataset Loading and Preparation

In [ ]:
load_dotenv()

entsoe_api_key = os.getenv("ENTSOE_API_KEY")

train_path = "data//train_data_2018_2023.parquet"
val_path = "data//val_data_2024.parquet"
test_path = "data//test_data_2025.parquet"

if not os.path.exists(train_path):
    # Training data from 2018 to 2023
    train_df = get_matched_weather_load_data(start_date=date(2018, 1, 1), end_date=date(2023, 12, 31), country_code="DE", locations=3, api_key=entsoe_api_key)
    train_df.to_parquet(train_path)
else:
    train_df = pd.read_parquet(train_path)

if not os.path.exists(val_path):
    # Validation data from 2024
    val_df = get_matched_weather_load_data(start_date=date(2024, 1, 1), end_date=date(2024, 12, 31), country_code="DE", locations=3, api_key=entsoe_api_key)
    val_df.to_parquet(val_path)
else:
    val_df = pd.read_parquet(val_path)

if not os.path.exists(test_path):
    # Test data from 2025
    test_df = get_matched_weather_load_data(start_date=date(2025, 1, 1), end_date=date(2025, 12, 31), country_code="DE", locations=3, api_key=entsoe_api_key)
    test_df.to_parquet(test_path)
else:
    test_df = pd.read_parquet(test_path)

# Features + Target (actual load data of the day in MW)
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [ ]:
scaler = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), SCALE_FEATURES)
    ],
    remainder="passthrough"
)

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "models/feature_scaler.joblib")

## Training

### 1.) Random Forest Regressor

In [ ]:
if not os.path.exists("models/rf_load_forecaster.joblib"):
    from sklearn.ensemble import RandomForestRegressor

    rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_regressor.fit(X_train_scaled, y_train)

    # evaluate_and_plot_model(rf_regressor, X_val_scaled, y_val, val_df)
    evaluate_and_plot_model(rf_regressor, X_test_scaled, y_test, test_df)

    joblib.dump(rf_regressor, "models/rf_load_forecaster.joblib")

### 2.) XGBoost Regressor

In [ ]:
if not os.path.exists("models/xgb_load_forecaster.joblib"):
    from xgboost import XGBRegressor

    xgb_regressor = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
    xgb_regressor.fit(X_train_scaled, y_train)

    # evaluate_and_plot_model(xgb_regressor, X_val_scaled, y_val, val_df)
    evaluate_and_plot_model(xgb_regressor, X_test_scaled, y_test, test_df)

    joblib.dump(xgb_regressor, "models/xgb_load_forecaster.joblib")